In [ ]:
# run this in terminal 
# pip install -r requirements.txt

^C


In [2]:
# ============================================================
# Output Directory
# ============================================================

import os

OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Output folder created")
print(os.path.abspath(OUTPUT_DIR))

✅ Output folder created
/Users/Abhi/Downloads/SOCS/model/outputs


In [3]:
import torch

print("Torch:", torch.__version__)
print("MPS Available:", torch.backends.mps.is_available())
print("MPS Built:", torch.backends.mps.is_built())

Torch: 2.12.1
MPS Available: True
MPS Built: True


In [4]:
from faster_whisper import WhisperModel

print("✅ Faster Whisper Imported")

✅ Faster Whisper Imported


In [5]:
import os
import json
import time
import warnings

import numpy as np
import torch
import librosa
import soundfile as sf
import noisereduce as nr

from faster_whisper import WhisperModel

warnings.filterwarnings("ignore")

print("✅ Libraries Imported Successfully")

✅ Libraries Imported Successfully


In [6]:
audio_file = "sample.mp3"



In [7]:
# ============================================================
# Load Audio
# ============================================================

audio, sr = librosa.load(
    audio_file,
    sr=None,
    mono=False
)

print("=" * 60)
print("Original Audio Information")
print("=" * 60)

print(f"Sample Rate : {sr} Hz")

if len(audio.shape) == 1:
    print("Channels    : Mono")
else:
    print(f"Channels    : {audio.shape[0]}")

duration = librosa.get_duration(
    y=audio,
    sr=sr
)

print(f"Duration    : {duration:.2f} sec")

print("=" * 60)

Original Audio Information
Sample Rate : 48000 Hz
Channels    : 2
Duration    : 1526.11 sec


In [8]:
# ============================================================
# Convert Stereo to Mono
# ============================================================

audio, sr = librosa.load(
    audio_file,
    sr=sr,
    mono=True
)

print("Converted to Mono")
print(audio.shape)

Converted to Mono
(73253376,)


In [9]:
# ============================================================
# Resample
# ============================================================

TARGET_SR = 16000

audio = librosa.resample(
    audio,
    orig_sr=sr,
    target_sr=TARGET_SR
)

sr = TARGET_SR

print(f"Resampled to {sr} Hz")

Resampled to 16000 Hz


In [10]:
# ============================================================
# Normalize Audio
# ============================================================

audio = librosa.util.normalize(audio)

print("Audio Normalized")

Audio Normalized


In [17]:
# ============================================================
# Noise Reduction
# ============================================================

reduced_audio = nr.reduce_noise(
    y=audio,
    sr=sr
)

print("Noise Reduction Complete")

Noise Reduction Complete


In [19]:
# ============================================================
# Save Processed Audio
# ============================================================

import os
import numpy as np
import soundfile as sf

processed_audio = os.path.join(
    OUTPUT_DIR,
    "processed_audio.wav"
)

reduced_audio = np.asarray(reduced_audio, dtype=np.float32)

sf.write(
    file=processed_audio,
    data=reduced_audio,
    samplerate=int(sr),
    format="WAV"
)

print("✅ Saved Successfully")
print(processed_audio)

✅ Saved Successfully
outputs/processed_audio.wav


Processed Audio Saved
outputs/processed_audio.wav


In [20]:
# ============================================================
# Audio Statistics
# ============================================================

print("=" * 60)

print("Processed Audio Summary")

print("=" * 60)

print(f"Sample Rate : {sr}")
print(f"Duration    : {len(reduced_audio)/sr:.2f} sec")
print(f"Max         : {np.max(reduced_audio):.4f}")
print(f"Min         : {np.min(reduced_audio):.4f}")
print(f"Mean        : {np.mean(reduced_audio):.4f}")

print("=" * 60)

Processed Audio Summary
Sample Rate : 16000
Duration    : 1526.11 sec
Max         : 0.6900
Min         : -0.7502
Mean        : -0.0000


In [13]:
# ============================================================
# Load Faster Whisper Model
# ============================================================

from faster_whisper import WhisperModel
import torch
import time
import platform

MODEL_SIZE = "base"
BATCH_SIZE = 8

# Device Configuration
if torch.cuda.is_available():
    device = "cuda"
    compute_type = "float16"
else:
    device = "cpu"
    compute_type = "int8"

print("=" * 60)
print("Loading Whisper Model...")
print("=" * 60)

start_time = time.time()

model = WhisperModel(
    MODEL_SIZE,
    device=device,
    compute_type=compute_type
)

end_time = time.time()

print("✅ Model Loaded Successfully")
print(f"Platform    : {platform.system()} {platform.machine()}")
print(f"Model       : {MODEL_SIZE}")
print(f"Device      : {device}")
print(f"Precision   : {compute_type}")
print(f"Load Time   : {end_time-start_time:.2f} sec")

Loading Whisper Model...


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-base/revision/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-base/tree/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Systran/faster-whisper-base/resolve/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Systran/faster-whisper-base/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66/tokenizer.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Systran/faster-whisper-base/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66/tokenizer.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/Systran/faster-whisper-base/resolve/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66/config.json "HTTP/1.1 307 

✅ Model Loaded Successfully
Platform    : Darwin arm64
Model       : base
Device      : cpu
Precision   : int8
Load Time   : 10.46 sec


In [22]:
# ============================================================
# Speech Recognition
# ============================================================

print("=" * 60)
print("Running Speech Recognition...")
print("=" * 60)

start_time = time.time()

segments, info = model.transcribe(
    processed_audio,
    beam_size=5,
    vad_filter=True,
    word_timestamps=True,
    condition_on_previous_text=True
)

end_time = time.time()

print("✅ Speech Recognition Complete")
print(f"Time Taken : {end_time-start_time:.2f} sec")

Running Speech Recognition...


INFO:faster_whisper:Processing audio with duration 25:26.112
INFO:faster_whisper:VAD filter removed 00:44.736 of audio
INFO:faster_whisper:Detected language 'en' with probability 0.92


✅ Speech Recognition Complete
Time Taken : 6.12 sec


In [23]:
# ============================================================
# Language Detection
# ============================================================

print("=" * 60)

print("Detected Language")

print("=" * 60)

print(f"Language      : {info.language}")

print(f"Confidence    : {info.language_probability:.2f}")

print("=" * 60)

Detected Language
Language      : en
Confidence    : 0.92


In [24]:
# ============================================================
# Build Transcript
# ============================================================

transcript = ""

segment_data = []

for segment in segments:

    transcript += segment.text.strip() + " "

    words = []

    if segment.words:

        for word in segment.words:

            words.append({

                "word": word.word,

                "start": round(word.start,2),

                "end": round(word.end,2),

                "probability": round(word.probability,3)

            })

    segment_data.append({

        "start": round(segment.start,2),

        "end": round(segment.end,2),

        "text": segment.text.strip(),

        "avg_logprob": segment.avg_logprob,

        "no_speech_probability": segment.no_speech_prob,

        "words": words

    })

print("=" * 60)

print(transcript)

print("=" * 60)

Hello and my name is Abhishek and welcome back to my channel. So today is day 3 of our DevOps 0 to 0 to 0 course. So if you haven't watched the previous videos, day 1 and day 2 and also day 0. So you can watch them on the DevOps playlist. So I have created a DevOps playlist and I'll keep uploading all this DevOps 0 to 0 videos on the same playlist. So you know you can go back and watch the playlist and then come back to the day 3. And today we are going to talk about virtual machines which is very very very important concept of DevOps. So you know if you don't know what is a server itself then we will also learn that if you don't know the concept of physical server we will also learn that today and we will try to learn everything with a real world example. Let's say you watched a lot of videos on virtual machines and you still don't understand what a virtual machine is. So today I will make sure that all of you will understand the concept of virtual machines with a real world example a

In [25]:
# ============================================================
# Save JSON
# ============================================================

speech_json = {

    "language": info.language,

    "language_probability": float(info.language_probability),

    "model": MODEL_SIZE,

    "transcript": transcript,

    "segments": segment_data

}

json_path = os.path.join(
    OUTPUT_DIR,
    "speech_output.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        speech_json,
        f,
        indent=4,
        ensure_ascii=False
    )

print("✅ JSON Saved")
print(json_path)

✅ JSON Saved
outputs/speech_output.json


In [26]:
# ============================================================
# Transcript Timeline
# ============================================================

print("=" * 70)

for seg in segment_data:

    print(
        f"[{seg['start']:>6} - {seg['end']:>6}] : {seg['text']}"
    )

print("=" * 70)

[   1.2 -   4.38] : Hello and my name is Abhishek and welcome back to my channel.
[  4.82 -  11.12] : So today is day 3 of our DevOps 0 to 0 to 0 course.
[ 11.66 -  16.18] : So if you haven't watched the previous videos, day 1 and day 2 and also day 0.
[ 16.56 -  19.22] : So you can watch them on the DevOps playlist.
[ 19.64 -  25.48] : So I have created a DevOps playlist and I'll keep uploading all this DevOps 0 to 0 videos
[ 25.48 -  26.68] : on the same playlist.
[ 26.68 -   30.7] : So you know you can go back and watch the playlist and then come back to the day 3.
[ 31.36 -  36.74] : And today we are going to talk about virtual machines which is very very very important
[ 36.74 -  37.88] : concept of DevOps.
[ 38.48 -   43.0] : So you know if you don't know what is a server itself then we will also learn that if you
[  43.0 -  47.24] : don't know the concept of physical server we will also learn that today and we will try
[ 47.24 -   49.4] : to learn everything with a real world ex

In [27]:
# ============================================================
# Speech Statistics
# ============================================================

print("=" * 60)

print("Speech Statistics")

print("=" * 60)

print(f"Segments          : {len(segment_data)}")

print(f"Language          : {info.language}")

print(f"Confidence        : {info.language_probability:.2f}")

print(f"Total Words       : {len(transcript.split())}")

print("=" * 60)

Speech Statistics
Segments          : 254
Language          : en
Confidence        : 0.92
Total Words       : 4350
